# 09 V4 — Final journal artifact QA/export

Duplicate-tolerant version. CPU only.

In [1]:

from pathlib import Path
import hashlib
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

print("=== V4 DUPLICATE-TOLERANT VERSION ===")

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_preferred(filename):
    hits = [p for p in INPUT_ROOT.rglob(filename) if p.is_file()]
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename!r} under /kaggle/input."
        )

    if len(hits) == 1:
        print("Found:", filename, "->", hits[0])
        return hits[0]

    # Kaggle can expose duplicated extracted folders, e.g.
    # ".../MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/" and "... (1)/".
    # If duplicates are byte-identical, safely select the cleanest path.
    hashes = {}
    for p in hits:
        hashes.setdefault(_sha256(p), []).append(p)

    if len(hashes) == 1:
        chosen = sorted(
            hits,
            key=lambda p: (
                " (1)" in str(p),
                " (2)" in str(p),
                len(str(p)),
                str(p),
            ),
        )[0]
        print(
            f"Found {len(hits)} identical copies of {filename}; "
            f"using: {chosen}"
        )
        return chosen

    # If copies are not identical, prefer the non-duplicated fixed-input path.
    preferred = [
        p for p in hits
        if "MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED" in str(p)
        and " (1)" not in str(p)
        and " (2)" not in str(p)
    ]
    if len(preferred) == 1:
        print(
            f"Warning: multiple non-identical copies of {filename}; "
            f"using the canonical fixed-input path: {preferred[0]}"
        )
        return preferred[0]

    raise RuntimeError(
        f"Found multiple non-identical copies of {filename!r}: {hits}. "
        "Please attach only one presentation-input dataset."
    )

def load_rgb(filename):
    return Image.open(find_preferred(filename)).convert("RGB")


=== V4 DUPLICATE-TOLERANT VERSION ===


In [2]:

markers = [
    "step2_analysis_metadata.json",
    "Table_locked_test_baseline_comparison.csv",
    "final_test_paired_statistics_all_metrics.csv",
    "ablation_oof_paired_statistics_all_metrics.csv",
    "baseline_oof_paired_statistics_all_metrics.csv",
    "reproducibility_audit.csv",
]
for name in markers:
    find_preferred(name)

print("\nInput check PASSED.")


Found 2 identical copies of step2_analysis_metadata.json; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/step2/step2_analysis_metadata.json
Found 2 identical copies of Table_locked_test_baseline_comparison.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/journal/tables/Table_locked_test_baseline_comparison.csv
Found 2 identical copies of final_test_paired_statistics_all_metrics.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/final_test_stats/final_test_paired_statistics_all_metrics.csv
Found 2 identical copies of ablation_oof_paired_statistics_all_metrics.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/ablation_stats/ablation_oof_paired_statistics_all_metrics.csv
Found 2 identical copies of b

In [3]:

OUT = WORK_ROOT / "MS3SEG_FINAL_JOURNAL_ARTIFACTS_V4"
if OUT.exists():
    shutil.rmtree(OUT)

for folder in ["figures", "tables", "statistics", "audit", "source_metadata"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

step2_marker = find_preferred("step2_analysis_metadata.json")
journal_table = find_preferred("Table_locked_test_baseline_comparison.csv")
final_stats = find_preferred("final_test_paired_statistics_all_metrics.csv")
ablation_stats = find_preferred("ablation_oof_paired_statistics_all_metrics.csv")
baseline_stats = find_preferred("baseline_oof_paired_statistics_all_metrics.csv")
audit_csv = find_preferred("reproducibility_audit.csv")

step2_root = step2_marker.parent
journal_root = journal_table.parent.parent
final_stats_root = final_stats.parent
ablation_stats_root = ablation_stats.parent
baseline_stats_root = baseline_stats.parent
audit_root = audit_csv.parent

def copy_matching(root, suffixes, destination, prefix):
    copied = []
    for source in Path(root).rglob("*"):
        if not source.is_file():
            continue
        if not any(source.name.lower().endswith(s.lower()) for s in suffixes):
            continue
        target = destination / f"{prefix}_{source.name}"
        n = 2
        while target.exists():
            target = destination / f"{prefix}_{source.stem}_{n}{source.suffix}"
            n += 1
        shutil.copy2(source, target)
        copied.append(target)
    return copied

copied = []
copied += copy_matching(journal_root, [".png", ".pdf"], OUT/"figures", "journal")
copied += copy_matching(journal_root, [".csv"], OUT/"tables", "journal")
copied += copy_matching(step2_root, [".png", ".pdf"], OUT/"figures", "step2")
copied += copy_matching(step2_root, [".csv"], OUT/"tables", "step2")
copied += copy_matching(final_stats_root, [".csv"], OUT/"statistics", "final_test_stats")
copied += copy_matching(ablation_stats_root, [".csv"], OUT/"statistics", "ablation_stats")
copied += copy_matching(baseline_stats_root, [".csv"], OUT/"statistics", "baseline_oof_stats")
copied += copy_matching(audit_root, [".csv", ".txt"], OUT/"audit", "reproducibility")
copied += copy_matching(step2_root, [".json"], OUT/"source_metadata", "step2")

print("Copied", len(copied), "artifacts.")


Found 2 identical copies of step2_analysis_metadata.json; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/step2/step2_analysis_metadata.json
Found 2 identical copies of Table_locked_test_baseline_comparison.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/journal/tables/Table_locked_test_baseline_comparison.csv
Found 2 identical copies of final_test_paired_statistics_all_metrics.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/final_test_stats/final_test_paired_statistics_all_metrics.csv
Found 2 identical copies of ablation_oof_paired_statistics_all_metrics.csv; using: /kaggle/input/datasets/saifsamiul/ms3seg-final-presentation-inputs-fixed/MS3SEG_FINAL_PRESENTATION_INPUTS_FIXED/ablation_stats/ablation_oof_paired_statistics_all_metrics.csv
Found 2 identical copies of b

In [4]:

rows = []
for path in sorted(p for p in OUT.rglob("*") if p.is_file()):
    raw = path.read_bytes()
    rows.append({
        "relative_path": str(path.relative_to(OUT)),
        "bytes": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(OUT / "artifact_manifest_sha256.csv", index=False)

required_patterns = {
    "OOF PR curve": "Figure13_OOF_precision_recall_curves",
    "OOF ROC curve": "Figure14_OOF_ROC_curves",
    "OOF reliability": "Figure15_OOF_reliability_diagram",
    "locked-test baseline table": "Table_locked_test_baseline_comparison",
    "OOF baseline table": "Table_OOF_baseline_comparison",
    "ablation table": "Table_ablation_OOF_comparison",
    "primary paired statistics": "Table_primary_paired_statistics",
    "reproducibility audit": "reproducibility_audit",
}

all_paths = "\n".join(manifest["relative_path"].tolist())
qa = pd.DataFrame([
    {"artifact": label, "pattern": pattern, "present": pattern in all_paths}
    for label, pattern in required_patterns.items()
])
qa.to_csv(OUT / "artifact_QA.csv", index=False)
display(qa)

if not qa["present"].all():
    missing = qa.loc[~qa["present"], "artifact"].tolist()
    raise RuntimeError(f"Missing required final artifacts: {missing}")

(OUT / "README.txt").write_text(
    "Final VentiMorph/MS3SEG journal-support artifact package. "
    "No model retraining, retuning, or test optimization was performed.",
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/kaggle/working/MS3SEG_FINAL_JOURNAL_ARTIFACTS_V4",
    "zip",
    root_dir=OUT,
)
print("QA PASSED")
print("Download:", archive)


,artifact,pattern,present
0,OOF PR curve,Figure13_OOF_precision_recall_curves,True
1,OOF ROC curve,Figure14_OOF_ROC_curves,True
2,OOF reliability,Figure15_OOF_reliability_diagram,True
3,locked-test baseline table,Table_locked_test_baseline_comparison,True
4,OOF baseline table,Table_OOF_baseline_comparison,True
5,ablation table,Table_ablation_OOF_comparison,True
6,primary paired statistics,Table_primary_paired_statistics,True
7,reproducibility audit,reproducibility_audit,True


QA PASSED
Download: /kaggle/working/MS3SEG_FINAL_JOURNAL_ARTIFACTS_V4.zip
